# 04 — Embeddings & Semantic Networks

This notebook explores semantic embeddings and semantic similarity networks.

It supports the project comparison between traditional keyword-based retrieval and semantic embedding-based retrieval.

## Goals

- Generate sentence/document embeddings.
- Run semantic retrieval for a project-relevant query.
- Build a semantic similarity network.
- Save semantic search and graph outputs.

In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()

for candidate in [current, *current.parents]:
    if (candidate / "src").exists() and (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find project root")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

from src.utils.common import (
    display_basic_frame_info,
    load_or_create_processed_documents,
    save_json,
    save_table,
    setup_notebook,
)

CONFIG, PATHS = setup_notebook()

## Load processed documents

In [ ]:
df = load_or_create_processed_documents(CONFIG, PATHS)

display_basic_frame_info(df, "Processed documents")

## Prepare document texts

In [ ]:
doc_ids = df["doc_id"].astype(str).tolist()
texts = df["text"].fillna("").astype(str).tolist()

print("Documents:", len(texts))
print("First document preview:")
print(texts[0][:1000])

## Configure embedding and retrieval settings

In [ ]:
embedding_cfg = CONFIG.get("embeddings", {})
retrieval_cfg = CONFIG.get("retrieval", {})

embedding_model_name = embedding_cfg.get("model", "all-MiniLM-L6-v2")
batch_size = embedding_cfg.get("batch_size", 32)
similarity_threshold = embedding_cfg.get("similarity_threshold", 0.5)

query = retrieval_cfg.get("example_query", "knowledge graph semantic retrieval")
top_k = retrieval_cfg.get("top_k", 10)

print("Embedding model:", embedding_model_name)
print("Batch size:", batch_size)
print("Similarity threshold:", similarity_threshold)
print("Query:", query)
print("Top-k:", top_k)

## Generate sentence embeddings

In [ ]:
from src.embeddings.sentence_embeddings import SentenceEmbedder

embedder = SentenceEmbedder(
    model_name=embedding_model_name,
    batch_size=batch_size,
)

embedder.fit(doc_ids, texts)
embeddings = embedder.embeddings

print("Embedding matrix shape:", embeddings.shape)

## Semantic retrieval example

In [ ]:
import numpy as np

query_embedding = embedder.encode([query])[0]

scores = np.array(
    [
        SentenceEmbedder.cosine_similarity(query_embedding, doc_embedding)
        for doc_embedding in embeddings
    ]
)

top_indices = np.argsort(scores)[::-1][:top_k]

semantic_results_df = pd.DataFrame(
    [
        {
            "rank": rank,
            "doc_id": doc_ids[index],
            "score": float(scores[index]),
            "text_preview": texts[index][:500],
        }
        for rank, index in enumerate(top_indices, start=1)
    ]
)

display(semantic_results_df)

## Save semantic retrieval output

In [ ]:
semantic_results_path = save_table(
    semantic_results_df,
    PATHS.data_processed / "notebook_semantic_search_results.csv",
)

print("Saved:", semantic_results_path)

## Build semantic similarity network

In [ ]:
from src.embeddings.network_embeddings import SemanticNetworkBuilder

network_builder = SemanticNetworkBuilder(
    similarity_threshold=similarity_threshold,
)

k = min(5, max(1, len(doc_ids) - 1))

semantic_graph = network_builder.build_knn_network(
    doc_ids=doc_ids,
    embeddings=embeddings,
    k=k,
)

semantic_stats = network_builder.network_stats(semantic_graph)
semantic_stats

## Save semantic network

In [ ]:
semantic_graph_path = PATHS.data_graphs / "notebook_semantic_network.graphml"
nx.write_graphml(semantic_graph, semantic_graph_path)

semantic_stats_path = save_json(
    semantic_stats,
    PATHS.data_processed / "notebook_semantic_network_summary.json",
)

print("Saved graph:", semantic_graph_path)
print("Saved stats:", semantic_stats_path)

## Visualize semantic network

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

# ------------------------------------------------------------
# 1. Select the most informative part of the semantic graph
# ------------------------------------------------------------

max_nodes = 30

top_nodes = sorted(
    semantic_graph.degree,
    key=lambda item: item[1],
    reverse=True,
)[:max_nodes]

selected_nodes = [node for node, _ in top_nodes]
subgraph = semantic_graph.subgraph(selected_nodes).copy()

print("Original graph nodes:", semantic_graph.number_of_nodes())
print("Original graph edges:", semantic_graph.number_of_edges())
print("Displayed graph nodes:", subgraph.number_of_nodes())
print("Displayed graph edges:", subgraph.number_of_edges())

# ------------------------------------------------------------
# 2. Create readable labels from paper titles
# ------------------------------------------------------------

title_lookup = dict(
    zip(
        df["doc_id"].astype(str),
        df["title"].fillna("").astype(str),
    )
)

labels = {
    node: title_lookup.get(str(node), str(node))[:45] + "..."
    if len(title_lookup.get(str(node), str(node))) > 45
    else title_lookup.get(str(node), str(node))
    for node in subgraph.nodes()
}

# ------------------------------------------------------------
# 3. Scale node sizes by graph degree
# ------------------------------------------------------------

degrees = dict(subgraph.degree())

node_sizes = [
    300 + degrees[node] * 80
    for node in subgraph.nodes()
]

# ------------------------------------------------------------
# 4. Scale edge thickness by similarity weight if available
# ------------------------------------------------------------

edge_weights = []

for _, _, data in subgraph.edges(data=True):
    weight = data.get("weight", data.get("similarity", 1.0))
    edge_weights.append(float(weight))

if edge_weights:
    min_weight = min(edge_weights)
    max_weight = max(edge_weights)

    if max_weight > min_weight:
        edge_widths = [
            0.5 + 3.0 * ((weight - min_weight) / (max_weight - min_weight))
            for weight in edge_weights
        ]
    else:
        edge_widths = [1.5 for _ in edge_weights]
else:
    edge_widths = 1.0

# ------------------------------------------------------------
# 5. Draw interpretable semantic network
# ------------------------------------------------------------

plt.figure(figsize=(18, 12))

pos = nx.spring_layout(
    subgraph,
    seed=CONFIG.get("project", {}).get("seed", 42),
    k=1.2,
)

nx.draw_networkx_edges(
    subgraph,
    pos,
    alpha=0.35,
    width=edge_widths,
)

nx.draw_networkx_nodes(
    subgraph,
    pos,
    node_size=node_sizes,
    alpha=0.9,
)

nx.draw_networkx_labels(
    subgraph,
    pos,
    labels=labels,
    font_size=8,
)

plt.title(
    "Top Semantic Similarity Network of Scientific Documents",
    fontsize=16,
)

plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
print(
    "Interpretation idea:\n"
    "- Dense graph regions indicate semantically similar groups of papers.\n"
    "- Weakly connected or isolated nodes may represent niche topics.\n"
    "- This network can later be compared with the knowledge graph built from extracted entities and relations."
)